# CineData Analytics | Landing → Bronze

**Objetivo:** ingerir os 5 CSVs brutos (Volume) e a cotação do dólar (API PTAX do Banco Central) na camada **Bronze**.

**Regras da camada Bronze aplicadas aqui:**
- Os dados são gravados **exatamente como chegam** (todas as colunas como `STRING`, sem renomear, sem limpar, sem deduplicar).
- Única coluna adicionada: `ingestion_datetime` (timestamp do momento da inserção).
- Formato **Delta**, modo **Append** (cada execução acrescenta um novo lote; a Silver decide qual versão é a mais recente).

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import time

import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

## Parâmetros (widgets)
- `catalogo`: catálogo do Unity Catalog onde ficam os schemas `bronze`, `silver` e `gold`.
- `caminho_volume`: pasta do Volume onde os 5 CSVs da pasta *Inputs* foram carregados (item 1.1).
- `data_inicio` / `data_fim`: janela da cotação no formato **MM-DD-AAAA**. Se ficarem vazios, usamos os
  **últimos 7 dias corridos** a partir da execução (a API não retorna fins de semana/feriados).

In [ ]:
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.text("caminho_volume", "/Volumes/workspace/landing/inputs", "Caminho do Volume (CSVs)")
dbutils.widgets.text("data_inicio", "", "Cotação - data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Cotação - data fim (MM-DD-AAAA)")

CATALOGO = dbutils.widgets.get("catalogo").strip()
CAMINHO_VOLUME = dbutils.widgets.get("caminho_volume").strip().rstrip("/")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Camada Bronze - dados brutos, sem transformação'")
print(f"Catálogo: {CATALOGO} | Volume: {CAMINHO_VOLUME}")

## 1. Ingestão dos CSVs
Leitura com `inferSchema = false`: tudo entra como texto, porque inferir tipo já seria uma transformação
(e, com os dados sujos, poderia descartar valores silenciosamente). `escape` trata aspas duplicadas dentro dos
campos e `multiLine` é ligado só onde há quebra de linha real dentro de aspas (sinopses do arquivo de info).

In [ ]:
# multiLine: só o arquivo de info tem sinopses com quebra de linha REAL dentro de aspas.
# Nos demais, cada linha física é um registro e existem aspas desbalanceadas (fruto do Column Shift):
# com multiLine=true o leitor "engoliria" milhares de linhas seguintes dentro de um único campo
# (ex.: metrics cairia de 107.364 para 103.072 registros). Por isso o parâmetro é definido por arquivo.
# O nome do arquivo usa curinga (*) porque o arquivo de info chega como movies_info_TMDB_IMDB.csv,
# diferente do nome citado no enunciado (movies_info_IMDB_TMDB.csv).
MAPEAMENTO_BRONZE = [
    # (padrão do arquivo,              tabela bronze,                  multiLine)
    ("movies_info_*.csv",              "bronze.tb_movies_info",        True),
    ("movies_financials_*.csv",        "bronze.tb_movies_financials",  False),
    ("movies_metrics_*.csv",           "bronze.tb_movies_metrics",     False),
    ("credits_and_tags_*.csv",         "bronze.tb_credits_and_tags",   False),
    ("movies_reviews*.csv",            "bronze.tb_movies_reviews",     False),
]


def ler_csv_bruto(caminho: str, multiline: bool):
    """Lê o CSV sem alterar estrutura nem conteúdo (todas as colunas como STRING)."""
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", str(multiline).lower())
        .option("quote", '"')
        .option("escape", '"')          # aspas dentro de campo vêm duplicadas ("") no padrão CSV
        .option("encoding", "UTF-8")
        .option("mode", "PERMISSIVE")   # linha malformada não derruba o pipeline
        .load(caminho)
    )


for padrao, tabela, multiline in MAPEAMENTO_BRONZE:
    df = ler_csv_bruto(f"{CAMINHO_VOLUME}/{padrao}", multiline)

    # Timestamp exato do momento da inserção deste lote na Bronze
    df = df.withColumn("ingestion_datetime", F.current_timestamp())

    df.write.format("delta").mode("append").saveAsTable(tabela)
    print(f"{padrao:<28} -> {tabela:<30} | {df.count():>7} linhas anexadas")

## 2. Ingestão da API do Banco Central (PTAX)
A diretoria financeira quer os valores também em BRL. Buscamos `dataHoraCotacao` e `cotacaoCompra`
e gravamos o retorno **cru** em `bronze.tb_cotacao_dolar` (append). O tratamento (série contínua com
forward fill) acontece na Silver.

In [ ]:
FUSO_BR = ZoneInfo("America/Sao_Paulo")
FORMATO_API = "%m-%d-%Y"  # formato exigido pela API: MM-DD-AAAA


def resolver_data(valor_widget: str, padrao: datetime) -> str:
    """Valida a data do widget (MM-DD-AAAA) ou usa o padrão quando vazio."""
    if not valor_widget.strip():
        return padrao.strftime(FORMATO_API)
    return datetime.strptime(valor_widget.strip(), FORMATO_API).strftime(FORMATO_API)  # erro claro se inválida


hoje = datetime.now(FUSO_BR)
data_fim = resolver_data(dbutils.widgets.get("data_fim"), hoje)
data_inicio = resolver_data(dbutils.widgets.get("data_inicio"), hoje - timedelta(days=7))
if datetime.strptime(data_inicio, FORMATO_API) > datetime.strptime(data_fim, FORMATO_API):
    raise ValueError(f"data_inicio ({data_inicio}) maior que data_fim ({data_fim})")

URL_PTAX = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    "?@dataInicial='{ini}'&@dataFinalCotacao='{fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)
url = URL_PTAX.format(ini=data_inicio, fim=data_fim)
print(f"Consultando PTAX de {data_inicio} a {data_fim}")


def chamar_api(url: str, tentativas: int = 3, espera_s: int = 5) -> list:
    """GET com retentativas simples: a API do BCB oscila com frequência."""
    for tentativa in range(1, tentativas + 1):
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            return resp.json().get("value", [])
        except Exception as erro:  # noqa: BLE001
            print(f"Tentativa {tentativa}/{tentativas} falhou: {erro}")
            if tentativa == tentativas:
                raise
            time.sleep(espera_s)


registros = chamar_api(url)
print(f"{len(registros)} cotações retornadas")

In [ ]:
SCHEMA_COTACAO = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True),
])

tabela_existe = spark.catalog.tableExists("bronze.tb_cotacao_dolar")

if registros:
    linhas = [(r.get("dataHoraCotacao"), float(r["cotacaoCompra"]) if r.get("cotacaoCompra") is not None else None)
              for r in registros]
    df_cotacao = (
        spark.createDataFrame(linhas, SCHEMA_COTACAO)
        .withColumn("ingestion_datetime", F.current_timestamp())
    )
    df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
    display(df_cotacao)
elif not tabela_existe:
    # Sem histórico nenhum a Silver não conseguiria converter para BRL: melhor falhar de forma explícita.
    raise RuntimeError("API sem cotações no período e bronze.tb_cotacao_dolar ainda não existe. Amplie a janela de datas.")
else:
    print("Nenhuma cotação nova no período (fim de semana/feriado). O histórico já existente será usado na Silver.")

## 3. Conferência da carga

In [ ]:
for tabela in [t for _, t, _ in MAPEAMENTO_BRONZE] + ["bronze.tb_cotacao_dolar"]:
    ultimo_lote = spark.table(tabela).agg(F.max("ingestion_datetime")).first()[0]
    print(f"{tabela:<30} | total: {spark.table(tabela).count():>8} | último lote: {ultimo_lote}")